# MURI: Modelling actuator self-assembly

## Imports statements and setup

First, import statements

In [ ]:
import numpy as np
import pyvista as pv
import polyscope as ps
import fabsim_py
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter
import IPython.display
import csv
import cv2
import igl

Constants

In [ ]:
dt = 1 / 24
k0 = 1e-4
k1 = 5e-2
kd = 0.1 # rate of dissociation
e0 = 1.2e-1
e1 = 1.7e-1
frac_f = 0.7
frac_s = 0.25
n = 16
world_coords_to_px = 1287.2

Fix DOFs

In [ ]:
fixed_dofs = []

def fix_dofs_in_circle(radius, center, V):
  for i in range(V.shape[0]):
    if np.linalg.norm(center - V[i, :2]) < radius + 1e-6:
      if center[0] * (center[0] - V[i, 0]) < 1:
        fixed_dofs.append(2 * i)
        fixed_dofs.append(2 * i + 1)

print(fixed_dofs)

Display compaction with polar plots

In [ ]:
def polygons_polar_plot(V, polymer_frac, scale):
  n = polymer_frac.shape[1]
  polygon = []
  rotated_polygon = []
  for i in range(n):
    angle = np.pi / n * i
    polygon.append(np.array([np.cos(angle), np.sin(angle)]))
    rotated_polygon.append(np.array([np.cos(angle + np.pi), np.sin(angle + np.pi)]))
  polygon = scale * np.array(polygon)
  rotated_polygon = scale * np.array(rotated_polygon)

  # faces = np.arange(F.shape[0] * 2 * n).reshape(F.shape[0], 2 * n)
  polygon_face = np.zeros((2 * n, 3))
  polygon_face[:, 1] = np.arange(1, 2 * n + 1)
  polygon_face[:, 2] = np.arange(2, 2 * n + 2)
  polygon_face[2 * n - 1, 2] = 1

  verts = []
  faces = []
  for i in range(F.shape[0]):
    center = (V[F[i, 0], :] + V[F[i, 1], :] + V[F[i, 2], :]) / 3
    verts.append(center.reshape((1, 2)))
    verts.append(polygon * polymer_frac[i, :][:, None] + center)
    verts.append(rotated_polygon * polymer_frac[i, :][:, None] + center)
    faces.append(polygon_face + i * (2 * n + 1))
  verts = np.concatenate(verts, axis=0)
  verts = np.hstack((verts, 0.1 * np.ones((verts.shape[0], 1))))
  faces = np.concatenate(faces, axis=0)

  return verts, faces

def vertex_polar_plot(V, polymer_frac, scale):
  n = polymer_frac.shape[1]
  polygon = []
  rotated_polygon = []
  for i in range(n):
    angle = np.pi / n * i
    polygon.append(np.array([np.cos(angle), np.sin(angle)]))
    rotated_polygon.append(np.array([np.cos(angle + np.pi), np.sin(angle + np.pi)]))
  polygon = scale * np.array(polygon)
  rotated_polygon = scale * np.array(rotated_polygon)

  # faces = np.arange(F.shape[0] * 2 * n).reshape(F.shape[0], 2 * n)
  polygon_face = np.zeros((2 * n, 3))
  polygon_face[:, 1] = np.arange(1, 2 * n + 1)
  polygon_face[:, 2] = np.arange(2, 2 * n + 2)
  polygon_face[2 * n - 1, 2] = 1

  verts = []
  faces = []
  for i in range(V.shape[0]):
    verts.append(V[i, :].reshape((1, 2)))
    verts.append(polygon * polymer_frac[i, :][:, None] + V[i, :])
    verts.append(rotated_polygon * polymer_frac[i, :][:, None] + V[i, :])
    faces.append(polygon_face + i * (2 * n + 1))
  verts = np.concatenate(verts, axis=0)
  verts = np.hstack((verts, 0.1 * np.ones((verts.shape[0], 1))))
  faces = np.concatenate(faces, axis=0)

  return verts, faces

In [ ]:
np.hstack((np.ones((10, 2)), 2 * np.ones((10, 1))))

CSV data processing

In [ ]:
def read_csv_to_array(filename):
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        data = [list(map(float, row)) for row in reader]
    return np.array(data, dtype=np.float32)

def to_8bit_rgb(x):
    R = np.floor(x)
    G = np.floor((x - R) * 256)
    B = np.floor(((x - R) * 256 - G) * 256)
    return R, G, B

def from_8bit_rgb(R, G, B):
    return R + G / 256. + B / 256. / 256.

## Polyscope init

In [ ]:
ps.init()
ps.set_give_focus_on_show(True)
ps.set_ground_plane_mode("shadow_only")

ps.set_navigation_style("planar")
ps.set_view_projection_mode("orthographic")
ps.set_SSAA_factor(3)
ps.set_view_from_json('{"farClipRatio":20.0,"fov":16.0,"nearClipRatio":0.005,"projectionMode":"Orthographic","viewMat":[1.0,-0.0,0.0,-0.0,0.0,0.997785151004791,-0.0665190145373344,0.000246047973632812,-0.0,0.0665190145373344,0.997785151004791,-25.5602951049805,0.0,0.0,0.0,1.0],"windowHeight":1964,"windowWidth":3456}')

Load initial mesh and display

In [ ]:
mesh = pv.read("data/top_surface.obj")
P = np.array(mesh.points, dtype=np.float64)
F = np.array(mesh.faces, dtype=np.int32)
F = F.reshape((F.shape[0] // 4, 4))[:, 1:]
P = P[:,:2]

ps_mesh = ps.register_surface_mesh("Initial mesh", P, F, smooth_shade=True, enabled=True, color=(42/255, 53/255, 213/255), edge_width=1)

fixed_dofs = []
X = np.reshape(P, 2 * P.shape[0])

fix_dofs_in_circle(0.75, np.array([-4.55/2, 0]), P)
X[fixed_dofs] += 2 * (4.55/2 - 2.2)
fix_dofs_in_circle(0.75, np.array([4.55/2, 0]), P)
X[fixed_dofs] -= (4.55/2 - 2.2)

P[:, 0] = np.reshape(X, P.shape)[:, 0]
fixed_dofs = np.sort(fixed_dofs)

ps.show()
ps.screenshot(filename='data/temp.png')
ps_mesh.set_enabled(False)
IPython.display.Image(filename='data/temp.png') 

In [ ]:
Phi = np.ones((F.shape[0], n))
P = P[:,:2]
V = P.copy()
stretch = 2
print(fabsim_py.model_hessian(V, P, F, Phi, fixed_dofs, stretch, 0.49, 1))
print(fabsim_py.model_hessian_finite_differences(V, P, F, Phi, fixed_dofs, stretch, 0.49, 1))

## Read SDF data

In [ ]:
read = cv2.imread('data/distance-boundary.png')
dist = from_8bit_rgb(read[:,:,0], read[:,:,1], read[:,:,2])

distance_grad_x = cv2.Sobel(dist, cv2.CV_64F, 1, 0, ksize=3, scale=1/8.)  # Gradient in X
distance_grad_y = cv2.Sobel(dist, cv2.CV_64F, 0, 1, ksize=3, scale=1/8.)  # Gradient in Y
dist_grad = world_coords_to_px * np.array([distance_grad_x, distance_grad_y])

cv2.imwrite('data/temp.png', dist)
IPython.display.Image(filename='data/temp.png')


In [ ]:
read = cv2.imread('data/distance-post.png')
dist2 = from_8bit_rgb(read[:,:,0], read[:,:,1], read[:,:,2])

distance_grad_x = cv2.Sobel(dist2, cv2.CV_64F, 1, 0, ksize=3, scale=1/8.)  # Gradient in X
distance_grad_y = cv2.Sobel(dist2, cv2.CV_64F, 0, 1, ksize=3, scale=1/8.)  # Gradient in Y
dist_grad2 = world_coords_to_px * np.array([distance_grad_x, distance_grad_y])

cv2.imwrite('data/temp.png', dist2)
IPython.display.Image(filename='data/temp.png')

In [ ]:
# P = P[:,:2]
# V = P.copy()

# indices = np.arange(V.shape[0])

# grad = fabsim_py.distance_gradient(V, indices, dist_grad[0], dist_grad[1])
# finite_diff = fabsim_py.distance_finite_differences(V, indices, dist)

# ps_mesh.add_vector_quantity("grad", np.reshape(grad, V.shape))
# ps_mesh.add_vector_quantity("finite_diff", np.reshape(finite_diff, V.shape))
# ps.show()

## Load fiber orientation and intensity data

In [ ]:
Phi = np.ones((F.shape[0], n))
P = P[:,:2]
V = P.copy()

for i in range(7):
  sigma = 0
  stretch = 1.25 + 0.25 * i
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

ps.remove_all_structures()
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')
# ps.show()

In [ ]:
img = cv2.imread('data/orientation_mean.png', cv2.IMREAD_UNCHANGED)
data = (256 - from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])) * np.pi / 256 # angles from the 'orientation.png' image have to be negated
data *= img[:,:,3] / 255

orientation = fabsim_py.orientation_data_to_mesh(np.concatenate((V, np.zeros_like(V[:,1:2])), axis=1), F, data, world_coords_to_px)
orientation /= np.max(np.linalg.norm(orientation, axis=1))
ps_mesh.add_vector_quantity("mean vector orientation", orientation, defined_on='faces', enabled=True)

ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
img = cv2.imread('data/orientation_dispersion.png', cv2.IMREAD_UNCHANGED)
data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * 0.5 / 256
data *= img[:,:,3] / 255

eta = fabsim_py.image_data_to_mesh(V, F, data, world_coords_to_px)
ps_mesh.add_scalar_quantity("eta", eta, defined_on='faces', enabled=True)

ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
read = cv2.imread('data/orientation.png', cv2.IMREAD_UNCHANGED)
orientation = from_8bit_rgb(read[:,:,0], read[:,:,1], read[:,:,2])
orientation = np.pi * (256. - orientation) / 256. # angles from the 'orentation.png' image have to be negated

Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation, read[:,:,3] / 255., world_coords_to_px, 0.3, n)

# normalize Phi
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16

verts, faces = vertex_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
Phi = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 

In [ ]:
verts, faces = polygons_polar_plot(P, Phi, 0.35)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", P, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 

In [ ]:
Phi = fabsim_py.update_Phi(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

verts, faces = polygons_polar_plot(P, Phi, 1)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", P, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 

In [ ]:
# distance_grad = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
sigma = 0
# test1 = fabsim_py.fiber_gradient(V, P, F, Phi, sigma)
# test2 = fabsim_py.fiber_finite_differences(V, P, F, Phi, sigma)

# ps_mesh.add_vector_quantity("test1", np.reshape(test1, V.shape))
# ps_mesh.add_vector_quantity("test2", np.reshape(test2, V.shape))

fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

H = fabsim_py.sensitivity_gradient_test2(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
df_dp = fabsim_py.model_gradient(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

# ps_mesh.add_vector_quantity("df/dp", np.reshape(df_dp, V.shape))

S = fabsim_py.sensitivity_matrix(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
# ps_mesh.add_vector_quantity("dx/d_stretch", np.reshape(S[:,0], V.shape))

# finite difference aproximation of "hessian"
dx_dp = -V.copy()
finite_diff = -fabsim_py.sensitivity_gradient_test1(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch + 1e-6, 0.49, sigma)

dx_dp += V.copy()
dx_dp /= 1e-6

finite_diff += fabsim_py.sensitivity_gradient_test1(V, P, F, Phi, fixed_dofs, stretch + 1e-6, 0.49, sigma)
finite_diff /= 1e-6

# ps_mesh.add_vector_quantity("dx/dp", dx_dp)
# ps_mesh.add_vector_quantity("df/dx * dx/dp", np.reshape(H.dot(np.reshape(dx_dp, df_dp.shape)), V.shape))
# ps_mesh.add_vector_quantity("df/dx * dx/d_stretch", np.reshape(H.dot(np.reshape(S[:,0], df_dp.shape)), V.shape))
ps_mesh.add_vector_quantity("residual", np.reshape(df_dp + H.dot(np.reshape(dx_dp, df_dp.shape)), V.shape))
ps_mesh.add_vector_quantity("finite diff", np.reshape(finite_diff, V.shape))
ps_mesh.add_vector_quantity("f", np.reshape(fabsim_py.sensitivity_gradient_test1(V, P, F, Phi, fixed_dofs, stretch + 1e-6, 0.49, sigma), V.shape))

# df/dp = model_gradient
# dx/dp = finite diff

ps.show()

In [ ]:
for i in range(10):
  stretch = 2.8
  sigma = 2 * i
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
  ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
  # ps.show()

  Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation, read[:,:,3] / 255., world_coords_to_px, 0.3, n)

  # normalize Phi
  Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * n
  Phi = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3

verts, faces = polygons_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))

fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
read = cv2.imread('data/intensity.png')
intensity = from_8bit_rgb(read[:,:,0], read[:,:,1], read[:,:,2])

I = fabsim_py.image_data_to_mesh(V, F, intensity, world_coords_to_px)
I = I / np.max(I)

ps.remove_all_structures()
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps_mesh.add_scalar_quantity("intensity", I, defined_on='faces', enabled=True)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
I = 0.025 * I / np.max(I)
print(np.sum(Phi, axis=1, keepdims=True) / n)
Phi = Phi / np.sum(Phi, axis=1, keepdims=True) * n
print(np.sum(Phi, axis=1, keepdims=True) / n)

for i in range(10):
  stretch = 2.77
  sigma = 100 * i
  fabsim_py.simulate_membrane(V, P, F, I[:,None] * Phi, fixed_dofs, stretch, 0.49, sigma)
  ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
  ps.show()

ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

## Sensitivity Analysis

In [ ]:
indices = igl.boundary_loop_all(F)
boundary_indices = indices[0]
post_indices = indices[1] + indices[2]

def distance(V, indices, dist, dist2):
  return 1.0 * fabsim_py.distance(V, indices[0], dist) + 1.0 * fabsim_py.distance(V, indices[1], dist2) + 1.0 * fabsim_py.distance(V, indices[2], dist2)

def distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2):
  grad = 1.0 * fabsim_py.distance_gradient(V, boundary_indices, dist_grad[0], dist_grad[1])
  grad += 1.0 * fabsim_py.distance_gradient(V, indices[1], dist_grad2[0], dist_grad2[1])
  grad += 1.0 * fabsim_py.distance_gradient(V, indices[2], dist_grad2[0], dist_grad2[1])
  return grad

In [ ]:
def fun(x):
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, x[0], 0.49, x[1])
  return distance(V, indices, dist, dist2)

print("Initial distance:", distance(V, indices, dist, dist2))

ret = minimize(fun, x0=np.array([2.94, 20]), method='Nelder-Mead')
print(ret)

Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation, read[:,:,3] / 255., world_coords_to_px, 0.3, n)
# normalize Phi
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * n
Phi = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3

verts, faces = polygons_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))

stretch = ret.x[0]
sigma = ret.x[1]
fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
print("Final distance:", distance(V, indices, dist, dist2))

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png')

In [ ]:
stretch = 2.92
sigma = 153
fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
d = distance(V, boundary_indices, post_indices, dist, dist2)
print(f"distance: {d}, stretch: {stretch}")

alpha = 1

for i in range(100):
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
  d = distance(V, boundary_indices, post_indices, dist, dist2)
  distance_grad = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
  g = fabsim_py.sensitivity_gradient(V, P, F, Phi, distance_grad, fixed_dofs, stretch, 0.49, sigma)

  # # finite difference aproximation of "hessian"
  # H = np.zeros((2, 2))
  # H[:, 0] = -distance_grad.dot(S)
  # H[:, 1] = -distance_grad.dot(S)
  # fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch + 1e-7, 0.49, sigma)
  # S_stretch = fabsim_py.sensitivity_matrix(V, P, F, Phi, fixed_dofs, stretch + 1e-7, 0.49, sigma)
  # distance_grad_stretch = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
  # H[:, 0] += distance_grad_stretch.dot(S_stretch)

  # fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma + 1e-7)
  # S_sigma = fabsim_py.sensitivity_matrix(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma + 1e-7)
  # distance_grad_sigma = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
  # H[:, 1] += distance_grad_sigma.dot(S_sigma)

  # H /= 1e-7
  # # print(H)

  # ds = -np.linalg.solve(H, distance_grad.dot(S))
  # # print(ds.dot(H) - distance_grad.dot(S))

  ds = -g

  if sigma + ds[1] < 0:
    ds[1] = -sigma
  if ds[0] > 1:
    ds[0] /= 100
  if ds[0] + stretch < 1:
    ds[0] = 1. - stretch

  # line search
  for _ in range(10):
    fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch + alpha * ds[0], 0.49, sigma + alpha * ds[1])
    new_d = distance(V, boundary_indices, post_indices, dist, dist2)
    if new_d < d:
      stretch += alpha * ds[0]
      sigma += alpha * ds[1]
      alpha *= 2
      d = new_d
      break
    else:
      print(f"line search: d={new_d} < {d}, ds={ds}")
      alpha /= 2
    if alpha < 1e-8:
      print("line search failed")
      break

  print(f"distance: {d}, stretch: {stretch}, sigma: {sigma}, step: {alpha}, ds: {ds}")

  if np.linalg.norm(ds) < 1e-6 or alpha < 1e-8 or stretch < 1 or sigma < 0 or sigma > 1000:
    break


ps.remove_all_structures()
mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
mesh.add_vector_quantity("distance_grad", np.reshape(distance_grad, V.shape))
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

In [ ]:
# Phi = np.zeros((F.shape[0], n))
# V = P.copy()

# ps.remove_all_structures()

sigmas = []
grads = []
hess = []
dists = []

for i in range(50):
  stretch = 2.8 + 0.002 * i
  # sigma = 0
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
  distance_grad = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
  S = fabsim_py.sensitivity_matrix(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

  # # finite difference aproximation of "hessian"
  # H = -distance_grad.dot(S[:,1])

  # fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma + 1e-4)
  # S = fabsim_py.sensitivity_matrix(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma + 1e-4)
  # distance_grad = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
 
  # H += distance_grad.dot(S[:,1])
  # H /= 1e-4

  # ds = -distance_grad.dot(S[:,1]) / H
  # print(f"distance: {fabsim_py.distance(V, boundary_indices, dist)}, sigma: {sigma}, gradient: {distance_grad.dot(S[:,1])}, ds: {ds}")

  sigmas.append(sigma)
  # grads.append(distance_grad.dot(S[:,0]))
  grads.append(np.linalg.norm(fabsim_py.model_gradient(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)))
  hess.append(fabsim_py.sensitivity_gradient(V, P, F, Phi, distance_grad, fixed_dofs, stretch, 0.49, sigma)[0])
  dists.append(fabsim_py.distance(V, boundary_indices, dist))

stretches = 2.8 + 0.002 * np.arange(50)

plt.figure(1)
plt.plot(stretches, dists)

plt.figure(2)
plt.subplot(121)
plt.plot(stretches, np.array(grads))
# plt.plot(stretches, np.array(hess))
# plt.show()

plt.subplot(122)
grads = np.array(grads)
finite_diff = grads[1:] - grads[0:-1]
plt.plot(stretches[1:], finite_diff)
# plt.plot(stretches[1:], np.array(grads)[1:] / finite_diff)
plt.show()


# ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
# ps.screenshot(filename='data/temp.png')
# IPython.display.Image(filename='data/temp.png')
# ps.show()

Sensitivity analysis with fiber parameter

In [ ]:
# ps.remove_all_structures()

# # convert Phi to face-based
# Phi = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3
stretch = 2.92
sigma = 200
# fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, 0)

boundary_indices = igl.boundary_loop(F)

d = fabsim_py.distance(V, boundary_indices, dist)
print(f"distance: {d}, sigma: {sigma}")

for i in range(10):
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)
  distance_grad = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
  S = fabsim_py.sensitivity_matrix(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

  # finite difference aproximation of "hessian"
  H = -distance_grad.dot(S[:,1])
  finite_diff = -fabsim_py.distance(V, boundary_indices, dist)

  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma + 1e-4)
  S = fabsim_py.sensitivity_matrix(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma + 1e-4)
  distance_grad = distance_gradient(V, boundary_indices, post_indices, dist_grad, dist_grad2)
 
  H += distance_grad.dot(S[:,1])
  H /= 1e-4
  finite_diff += fabsim_py.distance(V, boundary_indices, dist)
  finite_diff /= 1e-4

  print(f"gradient: {distance_grad.dot(S[:,1])}, finite diff: {finite_diff}")

  ds = -distance_grad.dot(S[:,1]) / H
  sigma += ds

  print(f"distance: {fabsim_py.distance(V, boundary_indices, dist)}, sigma: {sigma}, gradient: {distance_grad.dot(S[:,1])}, ds: {ds}")

  if np.abs(ds) < 1e-6:
    break

mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
mesh.add_vector_quantity("distance_grad", np.reshape(distance_grad, V.shape))
mesh.add_vector_quantity("S", np.reshape(S[:,1], V.shape))
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

## Fiber polymerization simulation

In [ ]:
stress = sigma * Phi
Phi_sim = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

verts, faces = polygons_polar_plot(V, Phi_sim, 0.25)
ps.register_surface_mesh("polymer frac simulated", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))

ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps_mesh.add_scalar_quantity("volume fraction", np.sum(Phi_sim, axis=1, keepdims=False) / n, defined_on='faces', enabled=True)

ps.show()

## Fitting the ODE coefficients for fiber simulation

In [ ]:
def fitting(phi_measured):
  def fun(params):
    k1 = params[0]
    kd = params[1]

    stress = sigma * Phi
    phi = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

    return (phi - phi_measured).flatten()

  initial_guess = np.array([5e-2, 0.1])
  return least_squares(fun, initial_guess, verbose=2)

In [ ]:
phi_measured = Phi
params = fitting(phi_measured)

print(k1, kd)

k1 = params.x[0]
kd = params.x[1]

print(k1, kd)

stress = sigma * Phi
phi_recovered = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

verts, faces = polygons_polar_plot(V, phi_recovered, 0.25)
ps.register_surface_mesh("polymer frac simulated", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))

ps_mesh.add_scalar_quantity("volume fraction", np.sum(phi_recovered, axis=1, keepdims=False) / n, defined_on='faces', enabled=True)

ps.show()

# Old

## Generate SDF data from masks

In [ ]:
from scipy.ndimage import distance_transform_edt

def generate_distance_from_mask(mask, show=False):
    mask = np.array(mask, dtype=bool)
    # Compute distance to the background (outside object)
    dist_outside= distance_transform_edt(mask, return_indices=False)

    # Compute distance to the foreground (inside object)
    dist_inside = distance_transform_edt(~mask, return_indices=False)

    # Signed Distance Field: negative inside, positive outside
    dist = 0.5 * (dist_outside - dist_inside)**2

    if show:
        plt.imshow(0.7769e-3 * (dist_inside - dist_outside), cmap='cividis')
        plt.colorbar(label='Signed Distance (mm)')
        plt.axis('off')
        plt.show()
    return dist

In [ ]:
n_cols = int(np.ceil(1287.2 * 4.225)) + 10
n_rows = int(np.ceil(1287.2 * 1.95)) + 10

dist = np.empty((n_rows, n_cols))
grad = np.empty((2, n_rows, n_cols))

for i in range(1,2):
  mask = cv2.imread(f"data/mask{i}-post.png", cv2.IMREAD_GRAYSCALE) < 128
  height, width = mask.shape
  
  mask1 = np.full((n_rows, n_cols), False)
  mask1[n_rows - height//2:n_rows, 0:width//2] += np.flip(mask[0:height//2, 0:width//2], axis=1)
  dist1 = generate_distance_from_mask(mask1, show=False)

  mask2 = np.full((n_rows, n_cols), False)
  mask2[n_rows - height//2:n_rows, 0:width//2] += np.flip(mask[height//2:height, 0:width//2], axis=(0, 1))
  dist2 = generate_distance_from_mask(mask2, show=False)

  mask3 = np.full((n_rows, n_cols), False)
  mask3[n_rows - height//2:n_rows, 0:width//2] += mask[0:height//2, width//2:width]
  dist3 = generate_distance_from_mask(mask3, show=False)

  mask4 = np.full((n_rows, n_cols), False)
  mask4[n_rows - height//2:n_rows, 0:width//2] += np.flip(mask[height//2:height, width//2:width], axis=0)
  dist4 = generate_distance_from_mask(mask4, show=False)

  dist += (dist1 + dist2 + dist3 + dist4) / 12

R, G, B = to_8bit_rgb(dist * 256 / np.max(dist))
img = np.array([R.T, G.T, B.T]).T
cv2.imwrite('data/distance-post.png', img)

## Steady-state solution for fiber distribution

In [ ]:
stretch_factor = 1.1
Phi = np.zeros((F.shape[0], n))
P = P[:,:2]
V = P.copy()

for i in range(7):
  sigma = 100
  stretch = 1.25 + 0.25 * i
  fabsim_py.simulate_membrane(V, P, F, Phi, fixed_dofs, stretch, 0.49, sigma)

  for i in range(3):
    Phi = fabsim_py.polymer_fraction_steady_state(sigma * Phi, 1, k1 / k0, kd / k0, frac_f, frac_s)
  ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

  verts, faces = polygons_polar_plot(V, Phi, 0.25)
  ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
  ps.show()


## Convergence of solutions

Convergence of ODE

In [ ]:
face_id = 392
stretch_factor = 1.5
polymer_frac = np.zeros((F.shape[0], n))
V = V.copy()
phi_t = [np.zeros(n)]

for i in range(1000):
  fabsim_py.simulate_membrane(V, V, F, polymer_frac, fixed_dofs, stretch_factor, 0.25)
  sigmas = 1.0 * fabsim_py.fiber_stress(V, V / stretch_factor, F, n)
  fabsim_py.polymer_fraction_one_step(polymer_frac, sigmas, k0, k1, kd, frac_f, frac_s, dt)
  phi_t.append(polymer_frac[face_id, :].copy())

phi_t = np.array(phi_t)

for i in range(n):
  plt.plot(phi_t[:, i])
plt.show()

Convergence of steady-state solution

In [ ]:
stretch_factor = 1.5
polymer_frac = np.zeros((F.shape[0], n))
V = P.copy()
phi_t = [np.zeros(n)]

for i in range(10):
  fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25, 1.0, e0, e1)
  stress = fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
  polymer_frac = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)
  phi_t.append(polymer_frac[face_id, :].copy())

phi_t = np.array(phi_t)

for i in range(n):
  plt.plot(phi_t[:, i])
plt.show()

Polar plot for "ground truth" solution

In [ ]:
stretch_factor = 1.5
n = 32
polymer_frac = np.zeros((F.shape[0], n))
V = P.copy()

for i in range(3):
  fabsim_py.simulate_membrane(V, P, F, polymer_frac, fixed_dofs, stretch_factor, 0.25)
  stress = fabsim_py.fiber_stress(V, P / stretch_factor, F, n, e0, e1)
  polymer_frac = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

_, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot([np.pi / n * i for i in range(3 * n)], np.tile(polymer_frac[face_id, :], 3))
ax.set_rmax(0.12)

plt.show()

Polar plot for fake "measured data" (= ground thruth + noise)

In [ ]:
phi_measured = polymer_frac + 0.01 * np.random.default_rng().random(polymer_frac.shape)

_, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot([np.pi / n * i for i in range(3 * n)], np.tile(phi_measured[face_id, :], 3))
ax.set_rmax(0.12)

plt.show()

In [ ]:
params = fitting(phi_measured)

print(k1, kd, e0, e1)

k1 = params.x[0]
kd = params.x[1]
e0 = params.x[2]
e1 = params.x[3]

print(k1, kd, e0, e1)

phi_recovered = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

_, ax = plt.subplots(subplot_kw={'projection': 'polar'})
ax.plot([np.pi / n * i for i in range(3 * n)], np.tile(phi_recovered[face_id, :], 3))
ax.set_rmax(0.12)

plt.show()

## Load orientation and intensity data

Load orientation CSV data

In [ ]:
input_csv = 'data/Orientation_angles_72hrs_compact_S1_16x_16x4_tilescan.csv' 
orientation_array = read_csv_to_array(input_csv)

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/orientation1-raw.png', img)
IPython.display.Image(filename='data/orientation1-raw.png')

In [ ]:
input_csv = 'data/orientation_angles_S2_72hrs_compact_16x_16x5_tilescan_denoise.csv'
orientation_array = read_csv_to_array(input_csv)
orientation_array = orientation_array

R, G, B = to_8bit_rgb(orientation_array * 256 / np.max(orientation_array))
A = np.clip(R + G + B, a_min=0, a_max=1) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
cv2.imwrite('data/orientation2-raw.png', img)
IPython.display.Image(filename='data/orientation2-raw.png')

In [ ]:
input_csv = 'data/orientation_angles_S3_72hrs_compact_16x_16x5_tilescan_denoise.csv'
orientation_array = read_csv_to_array(input_csv)
height, width = orientation_array.shape

# transfer to 4 channel image (to have transparency)
img = np.empty([height, width, 4])
img[:,:,0] = orientation_array
img[:,:,1] = np.zeros(orientation_array.shape)
img[:,:,2] = np.zeros(orientation_array.shape)
img[:,:,3] = np.clip(orientation_array, a_min=0, a_max=1e-6) * 255 / 1e-6

# Rotate image by 1 degree
rotation_matrix = cv2.getRotationMatrix2D(center=(width // 2, height // 2), angle=1, scale=1)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

# Add rotation angle to data
rotated_img[:,:,0] = np.mod(rotated_img[:,:,0] + np.pi / 180., np.pi)

# Convert to 8 bit RGB (aka BGR) format
R, G, B = to_8bit_rgb(rotated_img[:,:,0] * 256 / np.pi)
A = rotated_img[:,:,3]
rotated_img = np.array([R, G, B, A]).T
rotated_img = cv2.flip(rotated_img, 0)
 
cv2.imwrite('data/orientation3-raw.png', rotated_img)
IPython.display.Image(filename='data/orientation3-raw.png')

In [ ]:
img = cv2.imread(f'data/orientation.png', cv2.IMREAD_UNCHANGED)
mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
# array = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
# print(np.average(array * 360. / 256 - 180, weights=mask))
histr = cv2.calcHist([img],[0],mask,[256],[0,256])
plt.plot(histr)
plt.xlim([0,256])

for i in range(1,4):
    img = cv2.imread(f'data/orientation{i}.png', cv2.IMREAD_UNCHANGED)
    mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
    histr = cv2.calcHist([img],[0],mask,[256],[0,256])
    plt.plot(histr)
    plt.xlim([0,256])

plt.show()

In [ ]:
input_img = cv2.imread(f"data/orientation1.png")
height, width, _ = input_img.shape

orientation = np.zeros((height, width))
complex_a = np.zeros((height, width))
complex_b = np.zeros((height, width))

alpha = np.zeros((height, width))

for i in range(1,4):
  input_img = np.array(cv2.imread(f"data/orientation{i}.png", cv2.IMREAD_UNCHANGED), dtype=np.float32)
  input_img = input_img * (input_img[:,:,3:4] > 0)

  alpha += (input_img[:,:,3] > 0).astype(float)
  complex_a += np.cos(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)
  complex_b += np.sin(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)

complex_a /= np.clip(alpha, a_min=1, a_max=1000)
complex_b /= np.clip(alpha, a_min=1, a_max=1000)

complex_ab = np.sqrt(complex_a + 1j * complex_b)
orientation = np.mod(np.angle(complex_ab), np.pi) * 256 / np.pi

R, G, B = to_8bit_rgb(orientation)
A = (alpha > 0) * 255
img = np.array([R.T, G.T, B.T, A.T]).T

cv2.imwrite('data/orientation-combined.png', img)
IPython.display.Image(filename='data/orientation-combined.png')

In [ ]:
def add_mirror_images(target, source, source_flipped, height, width):
  n_rows = target.shape[0]

  target[n_rows - height//2:n_rows, 0:width//2] += np.flip(source_flipped[0:height//2, 0:width//2], axis=1)
  target[n_rows - height//2:n_rows, 0:width//2] += np.flip(source[height//2:height, 0:width//2], axis=(0, 1))
  target[n_rows - height//2:n_rows, 0:width//2] += source[0:height//2, width//2:width]
  target[n_rows - height//2:n_rows, 0:width//2] += np.flip(source_flipped[height//2:height, width//2:width], axis=0)

In [ ]:
n_cols = int(np.ceil(1287.2 * 4.225)) + 10
n_rows = int(np.ceil(1287.2 * 1.95)) + 10

alpha = np.zeros((n_rows, n_cols))
complex_a = np.zeros((n_rows, n_cols))
complex_b = np.zeros((n_rows, n_cols))

input_img = np.array(cv2.imread(f"data/orientation-combined.png", cv2.IMREAD_UNCHANGED), dtype=np.float32)

height, width, _ = input_img.shape

data = np.cos(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)
add_mirror_images(complex_a, data, data, height, width)

data = np.sin(2 * from_8bit_rgb(input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]) * np.pi / 256) * (input_img[:,:,3] > 0)
add_mirror_images(complex_b, data, -data, height, width)

data = (input_img[:,:,3] > 0).astype(float)
add_mirror_images(alpha, data, data, height, width)

complex_a /= np.clip(alpha, a_min=1, a_max=1000000)
complex_b /= np.clip(alpha, a_min=1, a_max=1000000)
complex_ab = np.sqrt(complex_a + 1j * complex_b)

orientation = np.mod(np.angle(complex_ab), np.pi) * 256 / np.pi
R, G, B = to_8bit_rgb(orientation)
A = (alpha > 0) * 255
orientation = np.array([R.T, G.T, B.T, A.T]).T

n_rows, n_cols, _ = orientation.shape
full_image = np.empty((2 * n_rows, 2 * n_cols, 4))

full_image[0:n_rows, 0:n_cols, :] = np.flip(orientation, axis=1)
full_image[n_rows:2*n_rows, 0:n_cols, :] = np.flip(orientation, axis=(0, 1))
full_image[0:n_rows, n_cols:2*n_cols, :] = orientation
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :] = np.flip(orientation, axis=0)

full_image[0:n_rows, 0:n_cols, :3] = (255 - full_image[0:n_rows, 0:n_cols, :3])
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :3] = (255 - full_image[n_rows:2*n_rows, n_cols:2*n_cols, :3])

cv2.imwrite('data/orientation.png', full_image)
IPython.display.Image(filename='data/orientation.png')

In [ ]:
n_cols = int(np.ceil(1287.2 * 4.225)) + 10
n_rows = int(np.ceil(1287.2 * 1.95)) + 10

orientation = np.full((n_rows, n_cols, 3), 0.0)
alpha = np.full((n_rows, n_cols, 1), 0.0)
for i in range(1,4):
  input_img = np.array(cv2.imread(f"data/orientation{i}.png", cv2.IMREAD_UNCHANGED), dtype=np.float32)
  img = input_img[:,:,:3] * input_img[:,:,3:4]
  img_flipped = (255 - input_img[:,:,:3]) * input_img[:,:,3:4]

  height, width, _ = input_img.shape
  
  orientation[n_rows - height//2:n_rows, 0:width//2] += np.flip(img_flipped[0:height//2, 0:width//2], axis=1)
  orientation[n_rows - height//2:n_rows, 0:width//2] += np.flip(img[height//2:height, 0:width//2], axis=(0, 1))
  orientation[n_rows - height//2:n_rows, 0:width//2] += img[0:height//2, width//2:width]
  orientation[n_rows - height//2:n_rows, 0:width//2] += np.flip(img_flipped[height//2:height, width//2:width], axis=0)

  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[0:height//2, 0:width//2, 3:4], axis=1)
  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[height//2:height, 0:width//2, 3:4], axis=(0, 1))
  alpha[n_rows - height//2:n_rows, 0:width//2] += input_img[0:height//2, width//2:width, 3:4]
  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[height//2:height, width//2:width, 3:4], axis=0)

orientation /= np.clip(alpha, a_min=1, a_max=1000000)

n_rows, n_cols, _ = orientation.shape
full_image = np.empty((2 * n_rows, 2 * n_cols, 4))

full_image[0:n_rows, 0:n_cols, :3] = (255 - np.flip(orientation, axis=1)) * np.flip(np.clip(alpha, a_min=0, a_max=1), axis=1)
full_image[n_rows:2*n_rows, 0:n_cols, :3] = np.flip(orientation, axis=(0, 1))
full_image[0:n_rows, n_cols:2*n_cols, :3] = orientation
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :3] = (255 - np.flip(orientation, axis=0)) * np.flip(np.clip(alpha, a_min=0, a_max=1), axis=0)
full_image[:,:,3] = np.clip(full_image[:,:,0] + full_image[:,:,1] + full_image[:,:,2], a_min=0, a_max=1) * 256

cv2.imwrite('data/orientation.png', full_image)
IPython.display.Image(filename='data/orientation.png')

Load pixel intensity CSV data

In [ ]:
input_csv = 'data/pixel_intensity_72hrs_compact_S1_16x_16x4_tilescan.csv'  # Replace with your filename
intensity_array = read_csv_to_array(input_csv)

R, G, B = to_8bit_rgb(intensity_array.T * 256 / np.max(intensity_array))
A = (R > 4) * 255
img = np.array([R, G, B, A]).T
# img = cv2.flip(intensity_array, 0)

# i, j = np.unravel_index(intensity_array.argmax(), intensity_array.shape)
# print(np.max(intensity_array))
# print(i, j)
# print(intensity_array[i-5:i+5,j-5:j+5])
cv2.imwrite('data/intensity1-raw.png', img)
IPython.display.Image(filename='data/intensity1-raw.png')

In [ ]:
data = cv2.imread('data/intensity1-16bit.png')
print(data[i-5:i+5,j-5:j+5, 0])


In [ ]:
input_csv = 'data/pixel_intensity_S2_72hrs_compact_16x_16x5_tilescan_denoise.csv'  # Replace with your filename
intensity_array = read_csv_to_array(input_csv)

R, G, B = to_8bit_rgb(intensity_array * 256 / np.max(intensity_array))
A = (R > 4) * 256
img = np.array([R, G, B, A]).T
img = cv2.flip(img, 0)
print(np.argmax(intensity_array))
cv2.imwrite('data/intensity2-raw.png', img)
IPython.display.Image(filename='data/intensity2-raw.png')

In [ ]:
input_csv = 'data/pixel_intensity_S3_72hrs_compact_16x_16x5_tilescan_denoise.csv'  # Replace with your filename
intensity_array = read_csv_to_array(input_csv)
height, width = intensity_array.shape

# transfer to 4 channel image (to have transparency)
img = np.empty([height, width, 4])
img[:,:,0] = 256 * intensity_array / np.max(intensity_array)
img[:,:,1] = np.zeros(intensity_array.shape)
img[:,:,2] = np.zeros(intensity_array.shape)
img[:,:,3] = (img[:,:,0] > 5) * 256

# Rotate image by 1 degree
rotation_matrix = cv2.getRotationMatrix2D(center=(width // 2, height // 2), angle=1, scale=1)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

# Convert to 8 bit RGB (aka BGR) format
R, G, B = to_8bit_rgb(rotated_img[:,:,0])
A = rotated_img[:,:,3]
rotated_img = np.array([R, G, B, A]).T
rotated_img = cv2.flip(rotated_img, 0)
 
cv2.imwrite('data/intensity3-raw.png', rotated_img)
print(np.argmax(intensity_array))
IPython.display.Image(filename='data/intensity3-raw.png')

In [ ]:
img = cv2.imread(f'data/intensity.png', cv2.IMREAD_UNCHANGED)
mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
# array = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])
# print(np.average(array * 360. / 256 - 180, weights=mask))
histr = cv2.calcHist([img],[0],mask,[256],[0,256])
plt.plot(histr)
plt.xlim([0,256])

for i in range(1,4):
    img = cv2.imread(f'data/intensity{i}.png', cv2.IMREAD_UNCHANGED)
    mask = (np.clip(img[:,:,3], a_min = 254, a_max=255) - 254) * 255
    histr = cv2.calcHist([img],[0],mask,[256],[0,256])
    plt.plot(histr)
    plt.xlim([0,256])

plt.show()

In [ ]:
n_cols = int(np.ceil(1287.2 * 4.225)) + 10
n_rows = int(np.ceil(1287.2 * 1.95)) + 10

intensity = np.full((n_rows, n_cols), 0.0)
alpha = np.full((n_rows, n_cols), 0.0)
for i in range(1,4):
  input_img = np.array(cv2.imread(f"data/intensity{i}.png", cv2.IMREAD_UNCHANGED), dtype=np.float32)
  img = input_img[:,:,:3] * input_img[:,:,3:4]
  data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2])

  height, width, _ = img.shape
  
  intensity[n_rows - height//2:n_rows, 0:width//2] += np.flip(data[0:height//2, 0:width//2], axis=1)
  intensity[n_rows - height//2:n_rows, 0:width//2] += np.flip(data[height//2:height, 0:width//2], axis=(0, 1))
  intensity[n_rows - height//2:n_rows, 0:width//2] += data[0:height//2, width//2:width]
  intensity[n_rows - height//2:n_rows, 0:width//2] += np.flip(data[height//2:height, width//2:width], axis=0)

  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[0:height//2, 0:width//2, 3], axis=1)
  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[height//2:height, 0:width//2, 3], axis=(0, 1))
  alpha[n_rows - height//2:n_rows, 0:width//2] += input_img[0:height//2, width//2:width, 3]
  alpha[n_rows - height//2:n_rows, 0:width//2] += np.flip(input_img[height//2:height, width//2:width, 3], axis=0)

intensity /= np.clip(alpha, a_min=1, a_max=1000000)
R, G, B = to_8bit_rgb(intensity)
A = (alpha > 0) * 255
intensity = np.array([R.T, G.T, B.T, A.T]).T

n_rows, n_cols, _ = intensity.shape
full_image = np.empty((2 * n_rows, 2 * n_cols, 4))

full_image[0:n_rows, 0:n_cols, :] = np.flip(intensity, axis=1)
full_image[n_rows:2*n_rows, 0:n_cols, :] = np.flip(intensity, axis=(0, 1))
full_image[0:n_rows, n_cols:2*n_cols, :] = intensity
full_image[n_rows:2*n_rows, n_cols:2*n_cols, :] = np.flip(intensity, axis=0)

cv2.imwrite('data/intensity.png', full_image)
IPython.display.Image(filename='data/intensity.png')

## Get mean and dispersion for orientation data

In [ ]:
# img = cv2.imread('data/orientation.png', cv2.IMREAD_UNCHANGED)

# data = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * np.pi / 256
# cos_data = np.cos(2 * data) * img[:,:,3]
# sin_data = np.sin(2 * data) * img[:,:,3]

# blur = 50

# # normalized convolution of image with mask
# sin_smooth = gaussian_filter(sin_data, sigma = blur)
# cos_smooth = gaussian_filter(cos_data, sigma = blur)

# weights = np.clip(gaussian_filter(img[:,:,3].astype(np.float64), sigma = blur), a_min=1, a_max=1000) 
# sin_smooth /= weights
# cos_smooth /= weights

# complex_ab = np.sqrt(cos_smooth + 1j * sin_smooth)

# mean_orientation = (np.angle(complex_ab) + np.pi / 2) * 256 / 255 * img[:,:,3] / np.pi

R, G, B = to_8bit_rgb(mean_orientation)
A = img[:,:,3]
new_img = np.array([R.T, G.T, B.T, A.T]).T

cv2.imwrite('data/orientation_mean.png', new_img)
IPython.display.Image(filename='data/orientation_mean.png')

In [ ]:
# mean resultant length
R = np.sqrt(sin_smooth**2 + cos_smooth**2)

kappa = np.zeros_like(R)

# approximation from Best & Fisher [1981]
lower_R = R[R < 0.53]
mid_R = R[(R > 0.53) & (R < 0.85)]
upper_R = R[R > 0.85]

kappa[R < 0.53] = 2 * lower_R + lower_R**3 + (5 * lower_R**5) / 6
kappa[(R > 0.53) & (R < 0.85)] = -0.4 + 1.39 * mid_R + 0.43 / (1 - mid_R)
kappa[R > 0.85] = 1 / (upper_R**3 - 4 * upper_R**2 + 3 * upper_R)

# print(256 / np.max(kappa[A > 0]))
r, g, b = to_8bit_rgb(kappa * 13)
A = img[:,:,3]

new_img = np.array([r.T, g.T, b.T, A.T]).T

cv2.imwrite('data/orientation_concentration*13.png', new_img)
IPython.display.Image(filename='data/orientation_concentration*13.png')

In [ ]:
import numpy as np
from numba import njit, prange

@njit(parallel=True, fastmath=True)
def integrate_field(P, x, dx):
    ny, nx = P.shape
    n_x = len(x)
    F = np.empty_like(P)

    for i in prange(ny):               # parallel loop over rows
        for j in range(nx):            # inner loop over columns
            p = P[i, j]
            acc = 0.0
            for k in range(n_x - 1):   # trapezoidal rule
                f1 = np.exp(p * np.cos(2 * x[k])) * np.sin(x[k])**2
                f2 = np.exp(p * np.cos(2 * x[k+1])) * np.sin(x[k+1])**2
                acc += 0.5 * (f1 + f2)
            F[i, j] = acc * dx
    return F

In [ ]:
import scipy.special

theta = np.linspace(-np.pi/2, np.pi/2, 256)

eta = integrate_field(kappa, theta, theta[1] - theta[0])
eta /= scipy.special.iv(0, kappa) * np.pi


In [ ]:
plt.plot(kappa.ravel(), eta.ravel())
plt.xlabel("$\kappa$")
plt.ylabel("$\eta$")
plt.show()

In [ ]:
r, g, b = to_8bit_rgb(eta * 256 / 0.5)
A = img[:,:,3]

new_img = np.array([r.T, g.T, b.T, A.T]).T

cv2.imwrite('data/orientation_dispersion.png', new_img)
IPython.display.Image(filename='data/orientation_dispersion.png')

In [ ]:
# Show image with HSV colormap
img = cv2.imread('data/orientation_mean.png', cv2.IMREAD_UNCHANGED)
mean_orientation = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * np.pi / 255
np.max(mean_orientation)

# plt.imshow(mean_orientation, cmap='twilight', origin='lower', extent=[0, mean_orientation.shape[1] / world_coords_to_px, 0, mean_orientation.shape[0] / world_coords_to_px], aspect='equal')
# plt.colorbar(label="Angle (radians)")
# plt.title("Mean tissue orientation")
# plt.tight_layout()
# plt.show()

In [ ]:
img = cv2.imread('data/orientation_concentration*13.png', cv2.IMREAD_UNCHANGED)
kappa = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) / 13

plt.imshow(kappa, cmap='turbo', origin='lower', extent=[0, kappa.shape[1] / world_coords_to_px, 0, kappa.shape[0] / world_coords_to_px], aspect='equal')
plt.colorbar(label="$\kappa$")
plt.title("Concentration measure $\kappa$")
plt.tight_layout()
plt.show()

In [ ]:
img = cv2.imread('data/orientation_dispersion.png', cv2.IMREAD_UNCHANGED)
eta = from_8bit_rgb(img[:,:,0], img[:,:,1], img[:,:,2]) * 0.5 / 256

plt.imshow(eta, cmap='turbo', origin='lower', extent=[0, mean_orientation.shape[1] / world_coords_to_px, 0, mean_orientation.shape[0] / world_coords_to_px], aspect='equal')
plt.colorbar(label="$\eta$")
plt.title("Dispersion measure $\eta$")
plt.tight_layout()
plt.show()

## Simulation comparisons

Shape from simulated fibers

In [ ]:
# Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.3, n)
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)

# normalize Phi
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16

verts, faces = vertex_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)

# ps.show()
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 

Shape from measured fibers

In [ ]:
# get orientation
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16

# sim
stretch_factor = 2.5
Phi_sim = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3
print(np.sum(Phi_sim))

fabsim_py.simulate_membrane(V, P, F, Phi_sim, fixed_dofs, stretch_factor, 0.49)

# display
verts, faces = vertex_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

Shape from measured fibers with intensity scaling

In [ ]:
# get intensity and orientation
intensity = fabsim_py.image_data_to_mesh(V, F, intensity_array, world_coords_to_px)
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)

# rescale intensity
intensity = intensity / np.max(intensity)

# rescale Phi
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16
Phi = Phi * intensity

# sim
stretch_factor = 2.5
Phi_sim = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3

fabsim_py.simulate_membrane(V, P, F, Phi_sim, fixed_dofs, stretch_factor, 0.49)

# display
verts, faces = vertex_polar_plot(V, Phi, 0.25)
ps.register_surface_mesh("polymer frac", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

## Fitting

In [ ]:
# get orientation
Phi = fabsim_py.histogram_data_to_mesh(V, F, orientation_array, world_coords_to_px, 0.6, n)
Phi = Phi / np.fmax(np.sum(Phi, axis=1, keepdims=True), 1) * 0.05 * 16

# sim
stretch_factor = 2.5
phi_measured = (Phi[F[:, 0]] + Phi[F[:, 1]] + Phi[F[:, 2]]) / 3

fabsim_py.simulate_membrane(V, P, F, phi_measured, fixed_dofs, stretch_factor, 0.49)

params = fitting(phi_measured)

print(k1, kd, e0, e1)

k1 = params.x[0]
kd = params.x[1]
e0 = params.x[2]
e1 = params.x[3]

print(k1, kd, e0, e1)

phi_recovered = fabsim_py.polymer_fraction_steady_state(stress, 1, k1 / k0, kd / k0, frac_f, frac_s)

fabsim_py.simulate_membrane(V, P, F, phi_recovered, fixed_dofs, stretch_factor, 0.49)

# display
verts, faces = polygons_polar_plot(V, phi_measured, 0.25)
ps.register_surface_mesh("phi_measured", verts, faces, enabled=False, color=(213/255, 202/255, 42/255))
verts, faces = polygons_polar_plot(V, phi_recovered, 0.25)
ps.register_surface_mesh("phi_recovered", verts, faces, enabled=True, color=(213/255, 202/255, 42/255))
ps_mesh = ps.register_surface_mesh("deformed", V, F, smooth_shade=True, color=(42/255, 53/255, 213/255), edge_width=1)
ps.screenshot(filename='data/temp.png')
IPython.display.Image(filename='data/temp.png') 
# ps.show()

## Tissue mesh extraction

Generate intensity mask

In [ ]:
mask = np.array(intensity_array > 300, dtype=float)

# Define a structuring element (kernel)
kernel = np.ones((25,25), np.uint8)

# Apply dilation
dilated_mask = cv2.dilate(mask, kernel, iterations=1)


cv2.imwrite('data/temp.png', dilated_mask * 255 / np.max(dilated_mask))
IPython.display.Image(filename='data/temp.png')

In [ ]:
mask = np.array(intensity_array2 > 300, dtype=float)

# Define a structuring element (kernel)
kernel = np.ones((25,25), np.uint8)

# Apply dilation
dilated_mask2 = cv2.dilate(mask, kernel, iterations=1)


cv2.imwrite('data/temp.png', dilated_mask2 * 255 / np.max(dilated_mask2))
IPython.display.Image(filename='data/temp.png')

In [ ]:
mask = np.array(intensity_array3 > 150, dtype=float)

# Define a structuring element (kernel)
kernel = np.ones((25,25), np.uint8)

# Apply dilation
dilated_mask3 = cv2.dilate(mask, kernel, iterations=1)

cv2.imwrite('data/temp.png', dilated_mask3 * 255 / np.max(dilated_mask3))
IPython.display.Image(filename='data/temp.png')